# Bet Sizing


Strategy: **sentiment_asynchronous**.

## Probability-Based Bet Sizing

- **Purpose:** Transform probabilities per event and average only within each security.
- **Settings:** Binary probability transform centered at 0.5; pass signals = 0; `K=10`; `step_size=0.10`; per-symbol weight cap = 5% of AUM.
- **Data:** Development predictions; the illustrative active set contains simultaneous event arrivals only.
- **Findings:**
- **Decision:** Apply the transform before averaging; apply discretization after averaging. The portfolio determines live activity from observed barriers.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths
from src.backtesting.portfolio_management import PortfolioSettings
PROJECT_ROOT = Path.cwd().resolve().parents[2]
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
HORIZON_BARS = 1000
K = 10
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment", period=PERIOD)
STRATEGY = "sentiment_asynchronous"
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
from src.backtesting.portfolio_management import load_prediction_events, prepare_calibration
from src.backtesting.bet_sizing import event_bet_signals, average_symbol_targets
events = load_prediction_events(paths, horizon_bars=HORIZON_BARS)
development = events.loc[events.partition.eq("development")].copy()
development["event_signal"] = event_bet_signals(development)
display(development[["symbol", "event_start", "event_signal"]].head())
example_time = development.event_start.iloc[0]
example_active = development.loc[development.event_start.eq(example_time)]
display(average_symbol_targets(example_active, settings.k, settings.step_size).to_frame())

## Price-Based Bet Sizing

- **Purpose:** Freeze per-symbol price calibration without holdout outcomes.
- **Settings:** Per-symbol development median of |entry price × target volatility|; calibration bet size = 0.5; downstream price sizing uses 100 steps.
- **Data:** Development event-start prices and target volatility.
- **Findings:**
- **Decision:** Persist calibration as a model artifact. Missing calibration excludes trading and is reported. No account simulation or event-return file is written.

In [ ]:
calibration = prepare_calibration(paths, manifest_path=MANIFEST_PATH, expected_securities=EXPECTED_SECURITIES, horizon_bars=HORIZON_BARS)
display(calibration)
display(calibration.loc[calibration.w.isna(), ["symbol", "reason"]])